# 03. 토큰 기반 청킹

`02_preprocess.ipynb`의 유효 문서를 모델 tokenizer 기준으로 분할합니다. 제목·카테고리 prefix와 특수 토큰을 포함한 최종 입력이 128토큰을 넘지 않는지 검사합니다.

In [1]:
from pathlib import Path
import json
import os
import tempfile

import pandas as pd
from IPython.display import display
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

def find_data_dir():
    for candidate in [Path.cwd(), Path.cwd() / 'ㅋㅌㅊ', Path.cwd().parent, Path.cwd().parent / 'ㅋㅌㅊ']:
        resolved = candidate.resolve()
        if (resolved / 'output/processed/maple_inven_tips_processed.json').is_file():
            return resolved
    raise FileNotFoundError('02_preprocess.ipynb를 먼저 실행하세요.')

DATA_DIR = find_data_dir()
OUTPUT_ROOT = DATA_DIR / 'output'
SETTINGS_PATH = OUTPUT_ROOT / 'intermediate/pipeline_settings.json'
PROCESSED_PATH = OUTPUT_ROOT / 'processed/maple_inven_tips_processed.json'
CHUNKS_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_documents_chunked.json'
settings = json.loads(SETTINGS_PATH.read_text(encoding='utf-8'))
MODEL_NAME = settings['model_name']
CHUNK_TOKENS = settings['chunk_tokens']
OVERLAP_TOKENS = settings['overlap_tokens']
MAX_TOKENS = settings['max_tokens']

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.model_max_length = 10**9  # 길이 측정 중 불필요한 경고만 방지
print('모델 tokenizer:', MODEL_NAME)
print(f'본문 목표={CHUNK_TOKENS}, 중첩={OVERLAP_TOKENS}, 최종 한도={MAX_TOKENS}')

C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모델 tokenizer: jhgan/ko-sroberta-multitask
본문 목표=100, 중첩=20, 최종 한도=128


In [2]:
def atomic_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write('\n')
            temporary = Path(stream.name)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def count_tokens(text, special_tokens=False):
    return len(tokenizer.encode(text, add_special_tokens=special_tokens, truncation=False))

def truncate_to_tokens(text, token_limit):
    if token_limit < 1:
        return ''
    if count_tokens(text) <= token_limit:
        return text
    low, high = 0, len(text)
    while low < high:
        middle = (low + high + 1) // 2
        if count_tokens(text[:middle]) <= token_limit:
            low = middle
        else:
            high = middle - 1
    return text[:low].rstrip()

def embedding_prefix(record, prefix_tokens):
    raw = f"문서 제목: {record['title']}\n카테고리: {record['category']}"
    return truncate_to_tokens(raw, prefix_tokens)

def build_embedding_text(chunk):
    candidate = f"{chunk['metadata']['embedding_prefix']}\n\n{chunk['page_content']}"
    token_count = count_tokens(candidate, special_tokens=True)
    if token_count > MAX_TOKENS:
        raise ValueError(f"임베딩 입력 토큰 제한 초과: {chunk['id']} ({token_count})")
    return candidate

def chunk_records(records):
    if CHUNK_TOKENS < 1:
        raise ValueError('CHUNK_TOKENS는 1 이상이어야 합니다.')
    if OVERLAP_TOKENS < 0 or OVERLAP_TOKENS >= CHUNK_TOKENS:
        raise ValueError('OVERLAP_TOKENS는 0 이상 CHUNK_TOKENS 미만이어야 합니다.')
    chunks = []
    for record in records:
        prefix_budget = max(1, MAX_TOKENS - CHUNK_TOKENS - 2)
        prefix = embedding_prefix(record, prefix_budget)
        body_budget = min(CHUNK_TOKENS, MAX_TOKENS - count_tokens(prefix) - 2)
        if body_budget < 1:
            raise ValueError(f"본문 토큰 예산이 없습니다: {record['document_id']}")
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=body_budget,
            chunk_overlap=min(OVERLAP_TOKENS, max(0, body_budget - 1)),
            length_function=count_tokens,
            separators=['\n\n', '\n', '. ', '? ', '! ', ' ', ''],
            is_separator_regex=False,
        )
        bodies = [body.strip() for body in splitter.split_text(str(record['content'])) if body.strip()]
        for chunk_index, body in enumerate(bodies):
            chunk_id = f"{record['document_id']}_{chunk_index}"
            metadata = {
                'source': 'guide', 'origin': 'inven_tip', 'source_name': '메이플 인벤 팁과 노하우',
                'document_id': record['document_id'], 'chunk_id': chunk_id, 'chunk_index': chunk_index,
                'article_id': record['article_id'], 'name': record['title'],
                'section_title': record['category'], 'url': record['url'],
                'created_at': record['created_at'], 'views': record['views'], 'likes': record['likes'],
                'text_quality': record['text_quality'], 'embedding_prefix': prefix,
            }
            chunk = {'id': chunk_id, 'page_content': body, 'metadata': metadata}
            build_embedding_text(chunk)
            chunks.append(chunk)
    if len({item['id'] for item in chunks}) != len(chunks):
        raise ValueError('중복 chunk_id가 생성됐습니다.')
    return chunks

In [3]:
processed = json.loads(PROCESSED_PATH.read_text(encoding='utf-8'))
chunks = chunk_records(processed)
atomic_write_json(CHUNKS_PATH, chunks)
token_counts = [count_tokens(build_embedding_text(chunk), special_tokens=True) for chunk in chunks]

display({
    '문서 수': len(processed),
    '청크 수': len(chunks),
    '고유 청크 ID': len({item['id'] for item in chunks}),
    '최소 입력 토큰': min(token_counts),
    '최대 입력 토큰': max(token_counts),
    '128토큰 초과': sum(count > MAX_TOKENS for count in token_counts),
    '저장 파일': str(CHUNKS_PATH),
})
preview = []
for chunk, token_count in zip(chunks[:5], token_counts[:5]):
    preview.append({
        'id': chunk['id'], 'category': chunk['metadata']['section_title'],
        'title': chunk['metadata']['name'], 'tokens': token_count,
        'page_content': chunk['page_content'][:220],
    })
display(pd.DataFrame(preview))

{'문서 수': 299,
 '청크 수': 5506,
 '고유 청크 ID': 5506,
 '최소 입력 토큰': 21,
 '최대 입력 토큰': 128,
 '128토큰 초과': 0,
 '저장 파일': 'C:\\Users\\Playdata\\Desktop\\team3_ 프로젝트1\\mle-01-p1-team3\\ㅋㅌㅊ\\output\\RAG\\maple_inven_tips_documents_chunked.json'}

,id,category,title,tokens,page_content
0,inven_tip_48082_0,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,114,메이플의 수많은 시스템들 중 뉴비가 접하기 쉽지 않거나 정보가 파편화 되어있어 알기...
1,inven_tip_48082_1,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,118,틀린 부분이 있다면 지적 부탁드립니다.\n제목 : 쿨타임 시스템\n부제 : 쿨뚝과 ...
2,inven_tip_48082_2,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,113,"스킬의 쿨타임이 줄어들면 그만큼 자주 사용할 수 있게 된다는 뜻이구요,\n좁게 보면..."
3,inven_tip_48082_3,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,114,"메르세데스 200레벨 기준 5%가 감소하며, 250레벨을 달성해야 6%가 감소합니다..."
4,inven_tip_48082_4,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,94,> 쿨타임 5% 감소가 큰가요? 별로 안커보이는데\n> 5%는 1200초(20분) ...


## 청킹 품질 진단

청킹 결과는 변경하지 않고 너무 짧거나 긴 청크, 완전 중복, 인접 청크의 과도한 유사성, 문맥 절단 의심 사례만 찾아봅니다. 의심 사례는 직접 확인

In [4]:
from collections import defaultdict
from difflib import SequenceMatcher
import hashlib
import re

def compact_text(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

body_token_counts = [count_tokens(chunk['page_content']) for chunk in chunks]
short_indices = [index for index, count in enumerate(body_token_counts) if count < 20]
over_limit_indices = [index for index, count in enumerate(token_counts) if count > MAX_TOKENS]

chunk_lookup = {chunk['id']: chunk for chunk in chunks}
hash_groups = defaultdict(list)
for chunk in chunks:
    normalized = compact_text(chunk['page_content'])
    digest = hashlib.sha256(normalized.encode('utf-8')).hexdigest()
    hash_groups[digest].append(chunk['id'])
exact_duplicate_groups = [ids for ids in hash_groups.values() if len(ids) > 1]
exact_duplicate_examples = [{
    'count': len(ids), 'ids': ids[:10],
    'content': chunk_lookup[ids[0]]['page_content'][:220],
} for ids in exact_duplicate_groups[:10]]

chunks_by_document = defaultdict(list)
for chunk in chunks:
    chunks_by_document[chunk['metadata']['document_id']].append(chunk)

adjacent_near_duplicates = []
suspicious_boundaries = []
closing_punctuation_pattern = re.compile(r'^[,.;:!?%)}〉】]')
delimiter_pairs = [('(', ')'), ('[', ']'), ('{', '}'), ('〈', '〉'), ('【', '】')]
for document_chunks in chunks_by_document.values():
    document_chunks.sort(key=lambda item: item['metadata']['chunk_index'])
    for previous, current in zip(document_chunks, document_chunks[1:]):
        previous_text = compact_text(previous['page_content'])
        current_text = compact_text(current['page_content'])
        similarity = SequenceMatcher(None, previous_text, current_text, autojunk=False).ratio()
        if similarity >= 0.90:
            adjacent_near_duplicates.append({
                'previous_id': previous['id'], 'current_id': current['id'],
                'similarity': round(similarity, 3),
                'previous_tail': previous_text[-100:], 'current_head': current_text[:100],
            })
        starts_with_closing_punctuation = bool(closing_punctuation_pattern.search(current_text))
        english_word_cut = bool(
            previous_text and current_text
            and previous_text[-1].isascii() and previous_text[-1].isalpha()
            and current_text[0].isascii() and current_text[0].islower()
        )
        unclosed_delimiter = any(
            previous_text.count(opening) > previous_text.count(closing) and closing in current_text[:100]
            for opening, closing in delimiter_pairs
        )
        if starts_with_closing_punctuation or english_word_cut or unclosed_delimiter:
            suspicious_boundaries.append({
                'previous_id': previous['id'], 'current_id': current['id'],
                'reason': (
                    '닫는 기호로 시작' if starts_with_closing_punctuation
                    else '영단어 중간 분리 의심' if english_word_cut
                    else '괄호·인용 구간이 청크 경계를 넘음'
                ),
                'previous_tail': previous_text[-100:], 'current_head': current_text[:100],
            })

quality_summary = {
    '전체 청크': len(chunks),
    '본문 토큰 최솟값': min(body_token_counts),
    '본문 토큰 중앙값': float(pd.Series(body_token_counts).median()),
    '본문 20토큰 미만': len(short_indices),
    '최종 128토큰 초과': len(over_limit_indices),
    '완전 중복 그룹': len(exact_duplicate_groups),
    '인접 유사도 0.9 이상': len(adjacent_near_duplicates),
    '문맥 절단 의심 경계': len(suspicious_boundaries),
}
display(quality_summary)
display(pd.DataFrame({
    'body_tokens': pd.Series(body_token_counts).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
}))

short_examples = [{
    'id': chunks[index]['id'], 'tokens': body_token_counts[index],
    'title': chunks[index]['metadata']['name'],
    'page_content': chunks[index]['page_content'][:220],
} for index in short_indices[:10]]
print('본문 20토큰 미만 샘플')
display(pd.DataFrame(short_examples))
print('완전 중복 그룹 샘플')
display(pd.DataFrame(exact_duplicate_examples))
print('인접 유사도 0.9 이상 샘플')
display(pd.DataFrame(adjacent_near_duplicates[:10]))
print('문맥 절단 의심 샘플')
display(pd.DataFrame(suspicious_boundaries[:10]))

{'전체 청크': 5506,
 '본문 토큰 최솟값': 3,
 '본문 토큰 중앙값': 91.0,
 '본문 20토큰 미만': 28,
 '최종 128토큰 초과': 0,
 '완전 중복 그룹': 25,
 '인접 유사도 0.9 이상': 3,
 '문맥 절단 의심 경계': 289}

,body_tokens
count,5506.000000
mean,85.478206
std,15.948839
min,3.000000
10%,65.000000
25%,81.000000
50%,91.000000
75%,96.000000
90%,99.000000
95%,100.000000


본문 20토큰 미만 샘플


,id,tokens,title,page_content
0,inven_tip_46595_9,15,메카베리 농장 경험치 분석,+ 레벨 별 총 경험치량 표 추가\n+ 경험치 배율 추가
1,inven_tip_46435_7,8,심해왕 공략 최종편 - 훈장작을 위한 타임어택 가이드,읽어주셔서 감사합니다.
2,inven_tip_46320_0,17,2025년 11월 기준 유니온 링크 표,여기에도 올립니다.\n2025년 11월 기준 메이플스토리 유니온 링크 표입니다.
3,inven_tip_46260_10,19,월드 아카이브 공략,[월드아카이브] 히든 일러스트 해금 조건\n궁금하신거 댓글달아주세요
4,inven_tip_45693_24,9,"(스압, 데이터주의) 전직업 시너지 효과표 2025ver","○\n샤프, 어블, 하울링"
5,inven_tip_44512_0,15,"앵글러 컴퍼니 3,7 스테이지",​\n윗키 + 로프커넥트 동시에 누르면 1개층 스킵가능
6,inven_tip_44052_0,12,방어구 세트효과 정리 미세팁,풀에테가 압살이긴하네요. ᄒᄒ
7,inven_tip_43800_0,19,스펙 구간별 결정 가격 증감률,10만 이상은 임의로 적은거라 차이가 있을 수 있습니다.
8,inven_tip_43068_0,14,(수정)썬데이/다음 이벤트 메할일&주요 보상 요약,[수정/추가 사항]\n1. 썬데이 등 주요 일정 추가
9,inven_tip_43068_36,12,(수정)썬데이/다음 이벤트 메할일&주요 보상 요약,4추만 보내주시면 감사드리겠습니다


완전 중복 그룹 샘플


,count,ids,content
0,15,"[inven_tip_47572_10, inven_tip_47572_16, inven...",----------------------------------------------...
1,2,"[inven_tip_47295_2, inven_tip_46627_3]","경매장에서 각 아이템 검색 후, 현재 가격을 입력해주시면 됩니다.\n아래\n30분 ..."
2,2,"[inven_tip_46261_11, inven_tip_45678_9]",----------\n3추만 보내주시면 감사드리겠습니다
3,2,"[inven_tip_46236_8, inven_tip_45500_12]",------------------------------------------
4,2,"[inven_tip_46034_2, inven_tip_40383_16]",----------------------------------------------...
5,2,"[inven_tip_45500_2, inven_tip_45500_8]",-----------------------------------------
6,2,"[inven_tip_44558_8, inven_tip_44524_7]",--------------------------------------\n1. 메인 ...
7,2,"[inven_tip_44558_9, inven_tip_44524_8]",▶ <피어나의 약초 바구니> 보약 이벤트에 SP 사용\n2. 메인 이벤트 주간 출석...
8,2,"[inven_tip_44558_10, inven_tip_44524_9]",- 레벨 범위 몬스터 1000마리 시 모래 1개 획득\nex) 하드 진 힐라 퇴치로...
9,2,"[inven_tip_44558_11, inven_tip_44524_10]",-\n월드당 코인 획득과 코인샵 이용이 가능합니다\n-----------------...


인접 유사도 0.9 이상 샘플


,previous_id,current_id,similarity,previous_tail,current_head
0,inven_tip_47118_24,inven_tip_47118_25,0.961,많은 메소와 스페어를 0.7개 더 요구하므로 사용할 이유가 없음 (다만 매우 적은...,"안 사용할 이유 또한 없다) 20성으로 자동복구시, 기댓값보다 0.94억 많은 메소..."
1,inven_tip_47118_25,inven_tip_47118_26,0.975,많은 메소와 스페어를 0.21개 더 요구하므로 사용할 이유가 없음 (다만 매우 적은...,"안 사용할 이유 또한 없다) 21성으로 자동복구시, 기댓값보다 0.91억 많은 메소..."
2,inven_tip_47118_26,inven_tip_47118_27,0.968,많은 메소와 스페어를 0.8개 더 요구하므로 사용할 이유가 없음 (다만 매우 적은...,"안 사용할 이유 또한 없다) 22성으로 자동복구시, 기댓값보다 0.32억 많은 메소..."


문맥 절단 의심 샘플


,previous_id,current_id,reason,previous_tail,current_head
0,inven_tip_48082_13,inven_tip_48082_14,닫는 기호로 시작,= 최종 쿨타임 위의 사진의 경우로 검증을 해봅시다. (15) x (1 - 5 / ...,) - 2 = 14.25 - 2 = 12.25 맞네요. > 와 그럼 6초 스킬에 5...
1,inven_tip_48082_18,inven_tip_48082_19,닫는 기호로 시작,쿨타임 이 식에 대입해보면 (10) x (1 - 5 / 100) x (1 - 3 ...,) x (0.85) = 9.5 x 0.85 = 8.075 제가 문과긴 한데 암튼 이...
2,inven_tip_47837_10,inven_tip_47837_11,괄호·인용 구간이 청크 경계를 넘음,"리한 표입니다. 결론적으로, 아이템 버닝과 최대한 유사하게라도 세팅을 하려면, [ ...","에픽 4셋 (모자, 상의, 하의) + 아케인 18성 유니크 / 에픽 5셋 (장갑, ..."
3,inven_tip_47805_20,inven_tip_47805_21,괄호·인용 구간이 청크 경계를 넘음,한참 뒤에 터지는 경우가 있다 최악의 경우 자폭이 권능이랑 겹쳐서 바인드에 걸리는 ...,메인 기믹 이름부터 'T-boy의 간섭') ​ 권능 파훼 이후 그로기에 걸렸을 때 ...
4,inven_tip_47572_12,inven_tip_47572_13,괄호·인용 구간이 청크 경계를 넘음,+ 핵심적으로 활용 가능한 유틸에는 임의로 별 표시를 해두었습니다. (해당 직업에 ...,챌섭에서 처음 키워볼 분들이 '이런 스킬로 버티면 되겠구나' 싶은 스킬들 위주로 선...
5,inven_tip_47515_19,inven_tip_47515_20,닫는 기호로 시작,"설치기와 꽤 높이 올라가는 스킬 윗점, 캔슬 돌진까지 가졌고 나쁘지 않은 유틸을 보...",. 라스트 스탠드가 어마어마한 비중을 차지하는 극딜 직업이기 때문에 극딜만 밀리지 ...
6,inven_tip_47485_6,inven_tip_47485_7,닫는 기호로 시작,"동안에는 어떠한 데미지도 없습니다. 50, 100, 150, 200 , 250 스...",", 영원한 꿈의 꽃 개화만 데미지가 있습니다. 데미지 점유율 [ 17.1% ] 꿈의..."
7,inven_tip_47485_8,inven_tip_47485_9,닫는 기호로 시작,"레버리, 페어리 더스트, 평딜이 나이트메어 (폭탄 마크)를 터트리며 나비 생성, ...",. 나이트메어는 최대 30개 까지만 누적됩니다. x30 부터는 나이트메어가 부여되지...
8,inven_tip_47443_7,inven_tip_47443_8,닫는 기호로 시작,"천하는 직업인 렌과 비교했을 때, 비슷한 수준의 유틸과 난이도, 조금 더 높은 딜량...",. 유니온 250작 필수 직업인 점 역시 250작을 많이 하기 어려운 뉴비에게 큰 ...
9,inven_tip_47443_13,inven_tip_47443_14,닫는 기호로 시작,"냥능력을 가진 캐릭터입니다. 뉴비 추천 직업에 들어가도 이상하지 않지만, 생각보다 ...",. 본섭 기준 보조가 저렴하다는 것이 여타 추천 직업들과의 차별점이 되겠습니다.
